# M21 - Executable safe-region and structural audit

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Submission evidence.** M21 is derived from the executable M20 outputs. No archived audit constants are used.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import tcr_core as tcr
REPRO=ROOT/'results'/'reproduced'; REPRO.mkdir(parents=True,exist_ok=True)

In [2]:
m20=pd.read_csv(REPRO/'m20a_replication_case_metrics.csv'); ap=pd.read_csv(REPRO/'m20b_replication_case_metrics.csv'); aq=pd.read_csv(REPRO/'m20c2_replication_case_metrics.csv')
def audit_row(study,d,eps):
    t=d[d.path=='temperature']; near=t.near_contained.mean(); matched=t.matched_near_rate.mean()
    return {'study':study,'epsilon_abs':eps,'n_cases':len(t),'mean_safe_width':t.safe_width.mean(),'width_fraction':t.safe_width_fraction.mean(),'near_containment':near,'matched_near_rate':matched,'near_containment_ratio':near/(matched+1e-12),'mean_safe_gain':t.safe_gain.mean(),'mean_full_gain':t.full_gain.mean(),'mean_safe_minus_matched_gain':t.safe_minus_matched_gain.mean(),'support_range':t.support_range.mean(),'spectral_radius_range':t.spectral_radius_range.mean()}
audit=pd.DataFrame([audit_row('Synthetic path comparison',m20,.005),audit_row('Appliances Energy',ap,.002),audit_row('Air Quality',aq,.002)]); audit.to_csv(REPRO/'m21_replication_summary.csv',index=False); display(audit.round(6))

,study,epsilon_abs,n_cases,mean_safe_width,width_fraction,near_containment,matched_near_rate,near_containment_ratio,mean_safe_gain,mean_full_gain,mean_safe_minus_matched_gain,support_range,spectral_radius_range
0,Synthetic path comparison,0.005,28,2.892857,0.222527,0.857143,0.362718,2.363112,0.002533,0.007031,0.027508,55.164510,0.171865
1,Appliances Energy,0.002,4,2.750000,0.211538,0.250000,0.361742,0.691099,0.006847,0.042316,-0.009347,45.690536,0.174737
2,Air Quality,0.002,4,2.750000,0.211538,0.500000,0.359848,1.389474,0.008563,0.016342,0.039252,45.515273,0.152232


In [3]:
# Synthetic tolerance sensitivity is re-evaluated from the frozen M20a task panel.
TASKS=['controlled_d1_clean','controlled_d20_clean','controlled_d20_white_plus_distractor','memory_d10','narma10','mackey_glass','lorenz_x']; CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=.8,ridge=1e-5)
rows=[]
for eps_rel in [.05,.10,.15,.20]:
    d=tcr.run_panel(TASKS,4,['temperature'],20260718,abs_tol=.005,rel_tol=eps_rel,**CONFIG)
    near=d.near_contained.mean(); matched=d.matched_near_rate.mean()
    rows.append({'epsilon_rel':eps_rel,'mean_safe_width':d.safe_width.mean(),'width_fraction':d.safe_width_fraction.mean(),'near_containment':near,'matched_near_rate':matched,'near_containment_ratio':near/(matched+1e-12),'mean_safe_gain':d.safe_gain.mean(),'mean_safe_minus_matched_gain':d.safe_minus_matched_gain.mean()})
tol=pd.DataFrame(rows); tol.to_csv(REPRO/'m21_tolerance_sensitivity.csv',index=False); display(tol.round(6))

,epsilon_rel,mean_safe_width,width_fraction,near_containment,matched_near_rate,near_containment_ratio,mean_safe_gain,mean_safe_minus_matched_gain
0,0.05,2.500000,0.192308,0.857143,0.307676,2.785865,0.002406,0.030778
1,0.10,2.892857,0.222527,0.857143,0.362718,2.363112,0.002533,0.027508
2,0.15,3.250000,0.250000,0.857143,0.402504,2.129524,0.002628,0.025443
3,0.20,3.428571,0.263736,0.857143,0.459047,1.867224,0.002628,0.024264


In [4]:
# Secondary top-quartile threshold summaries are derived from executable window tables.
def qstats(d,label):
    base=d.positive_safe_gain.mean(); out={'study':label,'base_positive_gain_rate':base}
    for col,prefix in [('safe_dispersion','dispersion'),('anchor_spread','anchor')]:
        w=d[np.isfinite(d[col])].copy(); q1,q3=w[col].quantile([.25,.75]); hi=w[w[col]>=q3]; lo=w[w[col]<=q1]; prec=hi.positive_safe_gain.mean(); out[prefix+'_high_precision']=prec; out[prefix+'_precision_ratio']=prec/(base+1e-12); out[prefix+'_high_minus_low_gain']=hi.safe_gain.mean()-lo.safe_gain.mean()
    return out
sec=pd.DataFrame([qstats(pd.read_csv(REPRO/'m19c_replication_window_metrics.csv'),'Synthetic indicator panel'),qstats(pd.read_csv(REPRO/'m20b_replication_window_metrics.csv'),'Appliances Energy'),qstats(pd.read_csv(REPRO/'m20c2_replication_window_metrics.csv'),'Air Quality')]); sec.to_csv(REPRO/'secondary_threshold_executable.csv',index=False); display(sec.round(6))

,study,base_positive_gain_rate,dispersion_high_precision,dispersion_precision_ratio,dispersion_high_minus_low_gain,anchor_high_precision,anchor_precision_ratio,anchor_high_minus_low_gain
0,Synthetic indicator panel,0.289474,0.395161,1.365103,0.017544,0.419355,1.448680,0.020626
1,Appliances Energy,0.653846,0.923077,1.411765,0.030768,0.948718,1.450980,0.030372
2,Air Quality,0.597826,0.739130,1.236364,0.026791,0.826087,1.381818,0.027992
